In [8]:
!pip install modal

In [9]:
!modal token new

Was not able to launch web browser
Please go to this URL manually and complete the flow:

]8;id=712980;https://modal.com/token-flow/tf-2IkIMjdi9SAdlMrVnGVXBO\https://modal.com/token-flow/tf-2IkIMjdi9SAdlMrVnGVXBO]8;;\

⠋ Waiting for authentication in the web browser
⠼ Waiting for token flow to complete...omplete...
Web authentication finished successfully!
Token is connected to the kpeach3010 workspace.
Verifying token against https://api.modal.com
Token verified successfully!
⠋ Storing token
Token written to /root/.modal.toml in profile kpeach3010.


In [10]:
import os

code = """
import modal

# 1. Định nghĩa môi trường (Image) - Giữ đúng các phiên bản của tác giả
catvton_image = (
    modal.Image.from_registry("nvidia/cuda:12.1.1-devel-ubuntu22.04", add_python="3.10")
    .apt_install(
        "git",
        "wget",
        "build-essential",
        "gcc",
        "g++",
        "clang",
        "ninja-build",
        "ffmpeg",
        "libgl1-mesa-glx",
        "libglib2.0-0",
        "unzip",
    )
    .env({
        "CUDA_HOME": "/usr/local/cuda",
        "CC": "gcc",
        "CXX": "g++"
    })
    .pip_install(
        "torch==2.1.2",
        "torchvision==0.16.2",
        "numpy==1.26.4",
        "cython",
        "pyyaml",
        "wheel",
        "setuptools",
        "tqdm",
        "av",
        "opencv-python",
        "pycocotools",
        "fvcore",
        "cloudpickle",
        "omegaconf",
        "huggingface_hub==0.26.2"
    )
    .run_commands(
    
        # cài build tools cần thiết
        "pip install wheel setuptools ninja",
    
        # clone detectron2
        "git clone https://github.com/facebookresearch/detectron2.git /tmp/detectron2",
    
        # build detectron2
        "cd /tmp/detectron2 && pip install -e . --no-build-isolation",
    
        # clone CatVTON
        "git clone https://github.com/Zheng-Chong/CatVTON.git /tmp/CatVTON",
    
        # copy code
        "cp -r /tmp/CatVTON/. /root/",
    
        # cleanup
        "rm -rf /tmp/CatVTON",
        "rm -rf /tmp/detectron2"
    )
    .pip_install(
        "accelerate==0.31.0",
        "transformers==4.46.3",
        "scipy==1.13.1",
        "diffusers==0.30.3",
        "fastapi",
        "uvicorn",
        "python-multipart",
        "pillow==10.3.0",
        "opencv-python==4.10.0.84"
    )
)

app = modal.App("catvton-api")
catvton_volume = modal.Volume.from_name("catvton-weights-volume", create_if_missing=True)

# 2. Class xử lý AI chạy trên GPU
@app.cls(gpu="T4", scaledown_window=180, image=catvton_image, volumes={"/weights": catvton_volume})
class CatVTONModel:
    @modal.enter()
    def load_model(self):
        import os, torch
        from huggingface_hub import snapshot_download
        from diffusers import DPMSolverMultistepScheduler
        from model.cloth_masker import AutoMasker
        from model.pipeline import CatVTONPipeline
        from utils import init_weight_dtype

        # 1. Định nghĩa đường dẫn trong Volume
        catvton_path = "/weights/zhengchong-catvton"
        sd_inpainting_path = "/weights/stable-diffusion-inpainting"

        # 2. Tải CatVTON weights (chỉ tải nếu thư mục trống)
        if not os.path.exists(catvton_path) or not os.listdir(catvton_path):
            snapshot_download(
                repo_id="zhengchong/CatVTON",
                local_dir=catvton_path,
                local_dir_use_symlinks=False
            )
        
        # 3. Tải Stable Diffusion Inpainting weights 
        if not os.path.exists(sd_inpainting_path) or not os.listdir(sd_inpainting_path):
            snapshot_download(
                repo_id="booksforcharlie/stable-diffusion-inpainting",
                local_dir=sd_inpainting_path,
                local_dir_use_symlinks=False
            )

        # 4. Khởi tạo Pipeline dùng đường dẫn local
        # Dùng trực tiếp path giúp Diffusers không cần check Hugging Face nữa
        self.pipeline = CatVTONPipeline(
            base_ckpt=sd_inpainting_path, 
            attn_ckpt=catvton_path,
            attn_ckpt_version="mix",
            weight_dtype=torch.float16,
            use_tf32=True,
            device="cuda"
        )

        # Nâng cấp Scheduler 
        if hasattr(self.pipeline, "scheduler"):
            self.pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
                self.pipeline.scheduler.config, 
                use_karras_sigmas=True, 
                algorithm_type="dpmsolver++"
            )

        # 5. Khởi tạo Masker dùng đường dẫn local
        self.automasker = AutoMasker(
            densepose_ckpt=os.path.join(catvton_path, "DensePose"),
            schp_ckpt=os.path.join(catvton_path, "SCHP"),
            device="cuda",
        )

    @modal.method()
    async def process_tryon(self, p_bytes, c_bytes, steps, cfg, seed):
        import io, torch
        from PIL import Image, ImageFilter
        from diffusers.image_processor import VaeImageProcessor
        from utils import resize_and_crop, resize_and_padding

        # 1. Tiền xử lý
        person = Image.open(io.BytesIO(p_bytes)).convert("RGB")
        cloth = Image.open(io.BytesIO(c_bytes)).convert("RGB")
        person = resize_and_crop(person, (768, 1024))
        cloth = resize_and_padding(cloth, (768, 1024))

        # 2. Tạo Mask
        mask_dict = self.automasker(person, "upper")
        mask = mask_dict['mask']

        # 3. Chạy Pipeline với Seed và Scheduler mới
        mask_blurred = VaeImageProcessor(vae_scale_factor=8, do_binarize=True, do_convert_grayscale=True).blur(mask, blur_factor=5)
        gen = torch.Generator(device="cuda").manual_seed(seed)
        
        raw_output = self.pipeline(
            image=person, condition_image=cloth, mask=mask_blurred,
            num_inference_steps=steps, guidance_scale=cfg, generator=gen
        )[0]

        # 4. Logic GIỮ NGUYÊN KHUÔN MẶT (Composite)
        soft_mask = mask.filter(ImageFilter.GaussianBlur(radius=3))
        final_img = Image.composite(raw_output, person, soft_mask)

        # 5. Giải phóng GPU & Trả kết quả JPEG
        torch.cuda.empty_cache()
        buf = io.BytesIO()
        final_img.save(buf, format='JPEG', quality=85)
        return buf.getvalue()

# 3. ASGI App để gọi từ Web
@app.function(image=catvton_image)
@modal.asgi_app()
def api():
    from fastapi import FastAPI, UploadFile, File, Form, Response
    web_app = FastAPI()

    @web_app.post("/tryon")
    async def tryon_endpoint(
        person_image: UploadFile = File(...),
        cloth_image: UploadFile = File(...),
        steps: int = Form(20),
        cfg: float = Form(5.0),
        seed: int = Form(42)
    ):
        model = CatVTONModel()
        p_b = await person_image.read()
        c_b = await cloth_image.read()
        
        # Truyền các tham số steps, cfg, seed vào model
        res = await model.process_tryon.remote.aio(p_b, c_b, steps, cfg, seed)
        
        return Response(content=res, media_type="image/jpeg")

    return web_app
  
"""

with open("app.py", "w") as f:
    f.write(code)

print("Đã tạo file app.py thành công!")

Đã tạo file app.py thành công!


In [11]:
!modal deploy app.py

⠸ Creating objects.....
⠦ Creating objects...ggle/working/app.py: Uploaded 0/1 files
⠇ Creating objects...ggle/working/app.py: Finalizing index of 1 files
├── 🔨 Created mount /kaggle/working/app.py
├── 🔨 Created web function api => https://kpeach3010--catvton-api-api.modal.run
└── 🔨 Created function CatVTONModel.*.
✓ Created objects.
├── 🔨 Created mount /kaggle/working/app.py
├── 🔨 Created web function api => https://kpeach3010--catvton-api-api.modal.run
└── 🔨 Created function CatVTONModel.*.
✓ App deployed in 0.946s! 🎉

View Deployment: https://modal.com/apps/kpeach3010/main/deployed/catvton-api


In [12]:
!modal volume ls catvton-weights-volume

        Directory listing of '/' in 'catvton-weights-volume'         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Filename                    ┃ Type ┃ Created/Modified     ┃ Size  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ zhengchong-catvton          │ dir  │ 2026-03-11 13:13 UTC │ 106 B │
│ stable-diffusion-inpainting │ dir  │ 2026-03-12 11:51 UTC │ 143 B │
└─────────────────────────────┴──────┴──────────────────────┴───────┘


In [7]:
!modal volume ls catvton-weights-volume zhengchong-catvton/

     Directory listing of 'zhengchong-catvton/' in 'catvton-weights-volume'     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Filename                             ┃ Type ┃ Created/Modified     ┃ Size    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ zhengchong-catvton/flux-lora         │ dir  │ 2026-03-11 13:13 UTC │ 32 B    │
│ zhengchong-catvton/vitonhd-16k-512   │ dir  │ 2026-03-11 13:13 UTC │ 9 B     │
│ zhengchong-catvton/mix-48k-1024      │ dir  │ 2026-03-11 13:13 UTC │ 9 B     │
│ zhengchong-catvton/dresscode-16k-512 │ dir  │ 2026-03-11 13:13 UTC │ 9 B     │
│ zhengchong-catvton/SCHP              │ dir  │ 2026-03-11 13:13 UTC │ 58 B    │
│ zhengchong-catvton/DensePose         │ dir  │ 2026-03-11 13:13 UTC │ 82 B    │
│ zhengchong-catvton/.cache            │ dir  │ 2026-03-11 13:13 UTC │ 11 B    │
│ zhengchong-catvton/config.json       │ file │ 2026-03-11 13:13 UTC │ 0 B     │
│ zhengchong-catvton/README.